<a href="https://colab.research.google.com/github/ansonkwokth/TableTennisPrediction/blob/dev/Siamese.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/ansonkwokth/TableTennisPrediction.git
%cd TableTennisPrediction

fatal: destination path 'TableTennisPrediction' already exists and is not an empty directory.
/content/TableTennisPrediction


In [2]:


import pandas as pd
from utils import data_loader as dl

import numpy as np
from model.Elo import Elo
from model.ModifiedElo import ModifiedElo
from model.ensemble import BaggingRatingSystem

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

import copy
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# Data

In [3]:
# GAME = 'TTStar'
# GAME = 'TTCup'
# GAME = 'SetkaCup'
GAME = 'SetkaCupWomen'
# GAME = 'LigaPro'


In [4]:
match GAME:
    case 'TTStar':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'TTCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCup':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'SetkaCupWomen':
        years = [2020, 2021, 2022, 2023, 2024]
    case 'LigaPro':
        years = [2022, 2023, 2024]
    case _:
        raise ValueError("Invalid game selected.")


text_data_game = dl.load_game_data(GAME, years, '../')
text_data = {
    year: text_data_game[year] for year in years
}
df = dl.create_game_dfs(GAME, years, text_data)

Loading ..//SetkaCupWomen2020.txt
Loading ..//SetkaCupWomen2021.txt
Loading ..//SetkaCupWomen2022.txt
Loading ..//SetkaCupWomen2023.txt
Loading ..//SetkaCupWomen2024.txt


In [5]:
# Generate ID indices for each pair of rows in the DataFrame
idx_lt = [i for i in range(len(df) // 2) for _ in range(2)]
df['ID'] = idx_lt  # Assign to the 'ID' column

# Reset the DataFrame index to ensure it's sequential
df.reset_index(drop=True, inplace=True)

# Get unique players and store them in player_lt
player_lt = df['Player'].unique()



In [6]:
year_val = years[-2]
year_test = years[-1]

df_train = df.loc[pd.DatetimeIndex(df['Date']).year < year_val]
df_val = df.loc[pd.DatetimeIndex(df['Date']).year == year_val]
df_train_val = df.loc[pd.DatetimeIndex(df['Date']).year <= year_val]
df_test = df.loc[pd.DatetimeIndex(df['Date']).year == year_test]

In [7]:
def format_to_array(df: pd.DataFrame) -> np.ndarray:

    # info_col = ['ID', 'Round', 'Datetime', 'Game', 'Date', 'Time']
    info_col = ['Round', 'Datetime', 'Game', 'Date', 'Time']
    col = [item for item in df.columns if item not in info_col]

    df[[c for c in col if "Set" in c]] = df[[c for c in col if "Set" in c]].astype(float)
    X = df[col].values.reshape(-1, 2, len(col))
    return X

In [8]:
X_train = format_to_array(df_train)
X_train_val = format_to_array(df_train_val)
X_val = format_to_array(df_val)
X_test = format_to_array(df_test)

In [9]:
X_all = format_to_array(df)

In [10]:
def get_data(X):
    data_all = []
    for game in X:
        game = game[:, 1:]
        player1, player2 = game[:, 0]
        scores = game[:, 1:]

        win1 = sum(scores[0]>scores[1])
        win2 = sum(scores[0]<scores[1])
        p1_win = int(win1 > win2)

        for si in scores.T:
            if (si[0] + si[1]) == 0: continue
            ti = si[0] / (si[0] + si[1])
            di = si[0] - si[1]
            if not np.isnan(ti):
                data_all.append([player1, player2, ti, di, p1_win])
    return data_all

def get_data_idx(data, player_to_idx):
    data_all_idx = []
    for game in data:
        data_all_idx.append((player_to_idx[game[0]], player_to_idx[game[1]], game[2], game[3] , game[4]))
    return data_all_idx

data_all = get_data(X_all)
data_train = get_data(X_train)
data_train_val = get_data(X_train_val)
data_test = get_data(X_test)
player_to_idx = {pn: i for i, pn in enumerate(np.unique(np.array(data_all)[:, :2]))}


data_all_idx = get_data_idx(data_all, player_to_idx)
data_train_idx = get_data_idx(data_train, player_to_idx)
data_train_val_idx = get_data_idx(data_train_val, player_to_idx)
data_test_idx = get_data_idx(data_test, player_to_idx)

In [11]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# # -----------------------------
# # Step 1: Simulate 10 players
# # -----------------------------
# num_players = 100
# player_names = [f"player{i}" for i in range(num_players)]
# players = {}
# for i in range(num_players):
#     # Each player gets a fixed Gaussian ability (mean and std)
#     mean = np.random.uniform(50, 100)
#     std = np.random.uniform(5, 15)
#     players[i] = {"name": player_names[i], "mean": mean, "std": std}


In [12]:

# # -----------------------------
# # Step 2: Simulate games and sets
# # -----------------------------
# def simulate_set(player1_id, player2_id):
#     """Simulate one set (first to 11 points) between two players."""
#     p1 = players[player1_id]
#     p2 = players[player2_id]
#     score1, score2 = 0, 0
#     while score1 < 11 and score2 < 11:
#         sample1 = np.random.normal(p1["mean"], p1["std"])
#         sample2 = np.random.normal(p2["mean"], p2["std"])
#         if sample1 > sample2:
#             score1 += 1
#         else:
#             score2 += 1
#     return score1, score2

# def simulate_game(player1_id, player2_id):
#     """Simulate a game (best of 5 sets: first to win 3 sets) between two players.
#        Returns a list of set results as tuples (score1, score2)."""
#     sets = []
#     wins1, wins2 = 0, 0
#     while wins1 < 3 and wins2 < 3:
#         s1, s2 = simulate_set(player1_id, player2_id)
#         sets.append((s1, s2))
#         if s1 > s2:
#             wins1 += 1
#         else:
#             wins2 += 1
#     return sets

# # Generate 100 games. For every set, record (player1_id, player2_id, t)
# # where t = (points of player1) / (total points in the set)
# data = []
# for _ in range(100):
#     player1_id, player2_id = np.random.choice(num_players, size=2, replace=False)
#     sets = simulate_game(player1_id, player2_id)
#     for s1, s2 in sets:
#         t = s1 / (s1 + s2)  # e.g., if the set ended 11:9 then t = 11/20 = 0.55
#         data.append((player1_id, player2_id, t))


In [18]:

# -----------------------------
# Step 3: Prepare the Dataset and DataLoader
# -----------------------------
class TableTennisDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        p1, p2, t, d, pb = self.data[idx]
        return int(p1), int(p2), t, d, pb

# dataset = TableTennisDataset(data)
# dataset = TableTennisDataset(data_all_idx)
dataset_train = TableTennisDataset(data_train_idx)
dataset_train_val = TableTennisDataset(data_train_val_idx)
dataloader_train = DataLoader(dataset_train, batch_size=32, shuffle=True)
dataloader_train_val = DataLoader(dataset_train_val, batch_size=32, shuffle=True)
# dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


In [19]:

# -----------------------------
# Step 4: Build the Integrated Siamese Network
# -----------------------------
class SiameseNetwork(nn.Module):
    def __init__(self, num_players, embedding_dim=8):
        super(SiameseNetwork, self).__init__()
        # Shared embedding layer for players.
        self.embedding = nn.Embedding(num_players, embedding_dim)
        # A small MLP to convert the embedding into a scalar "score."
        self.fc = nn.Sequential(
            # nn.Linear(embedding_dim, 16),
            # nn.ReLU(),
            # nn.Linear(16, 8),
            # nn.ReLU(),
            # nn.Linear(embedding_dim, 8),
            # nn.ReLU(),
            # nn.Linear(8, 4),
            # nn.ReLU(),
            nn.Linear(embedding_dim, 1)
        )

    def forward(self, player1_idx, player2_idx):
        # Process both players using the same embedding and MLP.
        embed1 = self.embedding(player1_idx)
        embed2 = self.embedding(player2_idx)
        score1 = self.fc(embed1).squeeze(-1)
        score2 = self.fc(embed2).squeeze(-1)
        # Directly return the win probability for player1.
        prob = 1.0 / (1.0 + torch.exp(score2 - score1))
        return prob


In [20]:

# -----------------------------
# Step 4: Build the Integrated Siamese Network
# -----------------------------
class SiameseNetwork2(nn.Module):
    def __init__(self, num_players, embedding_dim=8):
        super(SiameseNetwork2, self).__init__()
        # Shared embedding layer for players.
        self.embedding = nn.Embedding(num_players, embedding_dim)
        # A small MLP to convert the embedding into a scalar "score."
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim*2, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(4, 1)
        )

    def forward(self, player1_idx, player2_idx):
        # Process both players using the same embedding and MLP.
        embed1 = self.embedding(player1_idx)
        embed2 = self.embedding(player2_idx)
        combined1 = torch.cat([embed1, embed2], dim=1)
        combined2 = torch.cat([embed2, embed1], dim=1)
        y1 = self.fc(combined1).squeeze(-1)
        y2 = self.fc(combined2).squeeze(-1)
        # Directly return the win probability for player1.

        return nn.Sigmoid()(y1 - y2)


In [21]:
class SiameseNetwork3(nn.Module):
    def __init__(self, num_players, embedding_dim=8):
        super(SiameseNetwork3, self).__init__()
        # Shared embedding layer for players.
        self.embedding = nn.Embedding(num_players, embedding_dim)

        # Main branch to compute scalar "scores" for win probability.
        self.fc = nn.Sequential(
            nn.Linear(embedding_dim*2, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(4, 1)
        )

        # Auxiliary branch to predict the score difference.
        # It uses the concatenated embeddings from both players.
        self.fc_diff = nn.Sequential(
            nn.Linear(embedding_dim*2, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(16, 1)
        )

    def forward(self, player1_idx, player2_idx):
        # Embed both players.
        embed1 = self.embedding(player1_idx)
        embed2 = self.embedding(player2_idx)

        # Main branch: process both players in a symmetric fashion.
        combined1 = torch.cat([embed1, embed2], dim=1)
        combined2 = torch.cat([embed2, embed1], dim=1)
        y1 = self.fc(combined1).squeeze(-1)
        y2 = self.fc(combined2).squeeze(-1)

        # Win probability for player1.
        win_prob = torch.sigmoid(y1 - y2)

        # Auxiliary branch: predict score difference directly.
        # You could choose to use the same combined features or another formulation.
        score_diff_pred1 = self.fc_diff(torch.cat([embed1, embed2], dim=1)).squeeze(-1)
        score_diff_pred2 = self.fc_diff(torch.cat([embed2, embed1], dim=1)).squeeze(-1)
        score_diff_pred = score_diff_pred1 - score_diff_pred2

        return win_prob, score_diff_pred


In [40]:
def loss_fn(p, t):
    epsilon = 1e-7
    p = torch.clamp(p, epsilon, 1 - epsilon)
    loss = - (t * torch.log(p) + (1 - t) * torch.log(1 - p))
    return loss.mean()

def loss_fn(p, t, pb, lambda1=0.5):
    epsilon = 1e-7
    p = torch.clamp(p, epsilon, 1 - epsilon)
    loss1 = - (t * torch.log(p) + (1 - t) * torch.log(1 - p))
    loss2 = - (pb * torch.log(p) + (1 - pb) * torch.log(1 - p))
    loss = loss1 + loss2
    return loss.mean()


# def loss_fn(p, d, t, d_true, pb, lambda1=0.5, lambda2=0.005):
def loss_fn(p, d, t, d_true, pb, lambda1=0.5, lambda2=0.01):
    epsilon = 1e-7
    p = torch.clamp(p, epsilon, 1 - epsilon)
    loss1 = - (t * torch.log(p) + (1 - t) * torch.log(1 - p))
    loss2 = - (pb * torch.log(p) + (1 - pb) * torch.log(1 - p))
    loss3 = (d - d_true)**2
    loss = loss1 + lambda1 * loss2 + lambda2 * loss3
    # print(loss1.mean(), lambda1*loss2.mean())

    return loss.mean()

# def loss_fn(p, d, t, d_true, lambda1=0.005):
#     epsilon = 1e-7
#     p = torch.clamp(p, epsilon, 1 - epsilon)
#     loss1 = - (t * torch.log(p) + (1 - t) * torch.log(1 - p))
#     loss2 = (d - d_true)**2
#     loss = loss1 + lambda1 * loss2
#     # print(loss1.mean(), lambda1*loss2.mean())

#     return loss.mean()


num_players = len(player_to_idx)
# model = SiameseNetwork(num_players, embedding_dim=1)
# model = SiameseNetwork2(num_players, embedding_dim=8)
model = SiameseNetwork3(num_players, embedding_dim=8)
optimizer = optim.Adam(model.parameters(), lr=0.001)


num_epochs = 50
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in dataloader_train:
        # Use zip(*) to properly unpack the batch into separate tuples
        # player1_idx, player2_idx, t_val, d_val = batch
        player1_idx, player2_idx, t_val, d_val, pb_val = batch
        player1_idx = player1_idx.long()
        player2_idx = player2_idx.long()
        t_val = t_val.float()

        # p = model(player1_idx, player2_idx)
        p, d = model(player1_idx, player2_idx)
        # loss = loss_fn(p, (t_val>0.5).long())
        # loss = loss_fn(p, t_val, pb_val)
        # loss = loss_fn(p, t_val)
        # loss = loss_fn(p, d, t_val, d_val)
        loss = loss_fn(p, d, t_val, d_val, pb_val)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader_train):.4f}")



Epoch 1/50, Loss: 1.2470
Epoch 2/50, Loss: 1.2369
Epoch 3/50, Loss: 1.2297
Epoch 4/50, Loss: 1.2243
Epoch 5/50, Loss: 1.2201
Epoch 6/50, Loss: 1.2168
Epoch 7/50, Loss: 1.2146
Epoch 8/50, Loss: 1.2123
Epoch 9/50, Loss: 1.2110
Epoch 10/50, Loss: 1.2096
Epoch 11/50, Loss: 1.2091
Epoch 12/50, Loss: 1.2082
Epoch 13/50, Loss: 1.2079
Epoch 14/50, Loss: 1.2072
Epoch 15/50, Loss: 1.2072


KeyboardInterrupt: 

In [41]:

model.eval()  # set model to evaluation mode
correct = 0
total = 0

trained_players = np.unique(X_train[:, :, 1])

with torch.no_grad():
    for batch in X_val:

        player1, player2 = batch[:, 1]
        if player1 not in trained_players: continue
        if player2 not in trained_players: continue

        player1_idx, player2_idx = player_to_idx[player1], player_to_idx[player2]
        player1_idx = torch.tensor([player1_idx]).long()
        player2_idx = torch.tensor([player2_idx]).long()
        win1 = (sum(batch[0, 2:]>batch[1, 2:]))
        win2 = (sum(batch[0, 2:]<batch[1, 2:]))

        # Ground truth: 1 if player1 won the set, 0 otherwise.
        ground_truth = (win1 > win2)
        # Predicted probability from the model.
        # p = model(player1_idx, player2_idx)
        p, d = model(player1_idx, player2_idx)
        prediction = (p > 0.5).float()  # threshold at 0.5
        # print(int(prediction == ground_truth), prediction, ground_truth)

        correct += int(prediction == ground_truth)
        total += 1

accuracy = correct / total
print("Test accuracy: {:.2f}%".format(accuracy * 100))

Test accuracy: 59.76%


In [42]:
model(player1_idx, player2_idx), model(player2_idx, player1_idx)

((tensor([0.4918], grad_fn=<SigmoidBackward0>),
  tensor([-0.1756], grad_fn=<SubBackward0>)),
 (tensor([0.5082], grad_fn=<SigmoidBackward0>),
  tensor([0.1756], grad_fn=<SubBackward0>)))

In [ ]:
# model = SiameseNetwork(num_players, embedding_dim=1)
# model = SiameseNetwork2(num_players, embedding_dim=8)
model = SiameseNetwork3(num_players, embedding_dim=8)
optimizer = optim.Adam(model.parameters(), lr=0.001)


for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in dataloader_train_val:
        # Use zip(*) to properly unpack the batch into separate tuples
        player1_idx, player2_idx, t_val, d_val, pb_val = batch
        player1_idx = player1_idx.long()
        player2_idx = player2_idx.long()
        t_val = t_val.float()

        # p = model(player1_idx, player2_idx)
        p, d = model(player1_idx, player2_idx)
        # loss = loss_fn(p, t_val)
        loss = loss_fn(p, d, t_val, d_val, pb_val)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss/len(dataloader_train_val):.4f}")


In [ ]:

model.eval()  # set model to evaluation mode
correct = 0
total = 0

# trained_players = np.unique(X_train[:, :, 1])
trained_players = np.unique(X_train_val[:, :, 1])

with torch.no_grad():
    for batch in X_test:

        player1, player2 = batch[:, 1]
        if player1 not in trained_players: continue
        if player2 not in trained_players: continue

        player1_idx, player2_idx = player_to_idx[player1], player_to_idx[player2]
        player1_idx = torch.tensor([player1_idx]).long()
        player2_idx = torch.tensor([player2_idx]).long()
        win1 = (sum(batch[0, 2:]>batch[1, 2:]))
        win2 = (sum(batch[0, 2:]<batch[1, 2:]))

        # Ground truth: 1 if player1 won the set, 0 otherwise.
        ground_truth = (win1 > win2)
        # Predicted probability from the model.
        p, d = model(player1_idx, player2_idx)
        prediction = (p > 0.5).float()  # threshold at 0.5
        # print(int(prediction == ground_truth), prediction, ground_truth)
        # sdf
        correct += int(prediction == ground_truth)
        total += 1

accuracy = correct / total
print("Test accuracy: {:.2f}%".format(accuracy * 100))

In [ ]:
model(player1_idx, player2_idx) + model(player2_idx, player1_idx)